# Notebook 4 — National RSF-ready dataset (assembly only)

Ports the validated Massachusetts survival pipeline (`../ma_bridges/bridge_project.ipynb`
cleaning cell + `../ma_bridges/Build_RSF.ipynb`) to all 50 states + DC, then merges the
environmental features from Notebooks 2–3. **Scope: data assembly only — no model training.**

**Inputs** (all in `Bridges_all_of_US/`):
- `raw/{ST}_bridges.csv` — 51 per-state NBI bridge-year panels (~22.2M rows).
- `us_bridge_cell_map.csv` — `(STRUCTURE_NUMBER_008, STATE_FIPS) -> (grid_lat, grid_lon)`.
- `us_climate_by_cell_year.csv` — 18 climate features per `(grid_lat, grid_lon, SURVEY_YEAR)`.
- `us_bridge_coastal_distance.csv` — `(STRUCTURE_NUMBER_008, STATE_FIPS) -> dist_to_coast_km`.

**Outputs:**
- `us_rsf_data_a.csv` — one row per bridge (Option A): `(event, time)` + covariates.
- `us_rsf_data_clean.csv` — encoded model matrix, ready for `RandomSurvivalForest`
  (ALL dummy levels kept — trees are indifferent to the redundant column).
- `us_parametric_data_clean.csv` — same matrix with the first dummy of every one-hot
  group dropped (`drop_first=True` design), full-rank for parametric models
  (Cox/AFT, logistic, OLS) where the dummy-variable trap causes multicollinearity.

**Three national deviations from MA (all intentional):**
1. *Maintenance history* — MA used MA-DOT `maintenance.csv` (no 50-state equivalent).
   Replaced with NBI-native reconstruction proxies (`ever_reconstructed`,
   `years_since_reconstruction`) — which the MA pipeline already computed too.
2. *Geography* — `COUNTY_CODE_003` (~3,000 counties) / `HIGHWAY_DISTRICT_002`
   (state-specific) are dropped; geography is carried by `STATE_FIPS` one-hot +
   the continuous climate/coastal features. (MA's own `../ma_bridges/RSF.ipynb` already
   dropped every `COUNTY_` column at train time.)
3. *Climate normals, not per-year climate* — the 17 climate features are per-cell
   **1992–2025 means**. Option A keeps the first-event row for failures (median kept
   year 1996) but the last observed row for censored bridges (median 2025), so
   per-year values joined at the kept row's `SURVEY_YEAR` fingerprint the observation
   era and leak censoring status (measured: +0.011 test C-index on a national Cox arm,
   +0.09–0.13 on PA tree models — see `../Lu+Guler_comparison/`). For the same reason
   `years_since_reconstruction` is anchored to `REF_YEAR = 2025` and the raw
   `YEAR_RECONSTRUCTED_106` (now exactly collinear with it) is dropped.

Memory-safe: raw is processed **one state at a time** (a single state fits; the
full ~20M-row panel does not — established in Notebooks 2–3), collapsing each state to
one row per bridge before moving on.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

RAW_DIR   = Path("raw")
CELL_MAP  = Path("us_bridge_cell_map.csv")
CLIMATE   = Path("us_climate_by_cell_year.csv")
COASTAL   = Path("us_bridge_coastal_distance.csv")
OUT_A     = Path("us_rsf_data_a.csv")       # one row per bridge (Option A)
OUT_CLEAN = Path("us_rsf_data_clean.csv")   # encoded model matrix (ALL dummies -> RSF)
OUT_PARAM = Path("us_parametric_data_clean.csv")  # drop_first variant (parametric models)

KEYS = ["STATE_FIPS", "STRUCTURE_NUMBER_008"]     # structure # is unique only WITHIN a state
RARE_THRESHOLD = 200                              # min bridges for a categorical level to survive one-hot
CHUNK = 200_000                                   # raw rows per read chunk (TX is 1.8M rows; keeps peak RAM low)
REF_YEAR = 2025                                   # fixed anchor for years_since_reconstruction (era-free; see markdown)

# Raw NBI columns to load per state = the set the validated MA model carried
# (minus climate/maintenance, which we merge / replace). usecols keeps memory bounded.
LOAD_COLS = [
    "STRUCTURE_NUMBER_008", "STATE_FIPS", "SURVEY_YEAR",
    # --- event inputs (condition ratings) + leakage cols dropped later ---
    "DECK_COND_058", "SUPERSTRUCTURE_COND_059", "SUBSTRUCTURE_COND_060",
    "CHANNEL_COND_061", "CULVERT_COND_062",
    # --- age / reconstruction ---
    "YEAR_BUILT_027", "YEAR_RECONSTRUCTED_106",
    # --- geography (raw; county/district dropped in encoding, lat/lon kept like MA) ---
    "HIGHWAY_DISTRICT_002", "COUNTY_CODE_003", "PLACE_CODE_004", "LAT_016", "LONG_017",
    "KILOPOINT_011", "BASE_HWY_NETWORK_012", "LRS_INV_ROUTE_013A", "SUBROUTE_NO_013B",
    "MIN_VERT_CLR_010",
    # --- administrative (loaded to mirror MA, dropped in encoding) ---
    "RECORD_TYPE_005A", "CRITICAL_FACILITY_006B",
    # --- structural / operational features (survive into the model) ---
    "DETOUR_KILOS_019", "TOLL_020", "MAINTENANCE_021", "OWNER_022", "FUNCTIONAL_CLASS_026",
    "TRAFFIC_LANES_ON_028A", "TRAFFIC_LANES_UND_028B", "ADT_029", "YEAR_ADT_030",
    "DESIGN_LOAD_031", "APPR_WIDTH_MT_032", "MEDIAN_CODE_033", "DEGREES_SKEW_034",
    "STRUCTURE_FLARED_035", "RAILINGS_036A", "TRANSITIONS_036B", "APPR_RAIL_036C",
    "APPR_RAIL_END_036D", "HISTORY_037", "NAVIGATION_038", "OPEN_CLOSED_POSTED_041",
    "SERVICE_ON_042A", "SERVICE_UND_042B", "STRUCTURE_KIND_043A", "STRUCTURE_TYPE_043B",
    "APPR_KIND_044A", "APPR_TYPE_044B", "MAIN_UNIT_SPANS_045", "APPR_SPANS_046",
    "HORR_CLR_MT_047", "MAX_SPAN_LEN_MT_048", "STRUCTURE_LEN_MT_049", "LEFT_CURB_MT_050A",
    "RIGHT_CURB_MT_050B", "ROADWAY_WIDTH_MT_051", "DECK_WIDTH_MT_052", "VERT_CLR_OVER_MT_053",
    "VERT_CLR_UND_REF_054A", "VERT_CLR_UND_054B", "LAT_UND_REF_055A", "LAT_UND_MT_055B",
    "LEFT_LAT_UND_MT_056", "OPERATING_RATING_064", "INVENTORY_RATING_066", "DATE_OF_INSPECT_090",
    "STRAHNET_HIGHWAY_100", "TRAFFIC_DIRECTION_102", "TEMP_STRUCTURE_103", "HIGHWAY_SYSTEM_104",
    "FEDERAL_LANDS_105", "DECK_STRUCTURE_TYPE_107", "SURFACE_TYPE_108A", "MEMBRANE_TYPE_108B",
    "DECK_PROTECTION_108C", "PERCENT_ADT_TRUCK_109", "NATIONAL_NETWORK_110", "PIER_PROTECTION_111",
    "BRIDGE_LEN_IND_112", "SCOUR_CRITICAL_113", "DECK_AREA",
]

# Columns that carry non-numeric codes (e.g. 'N') -> read as string so we control coercion.
STR_COLS = {c: str for c in [
    "STRUCTURE_NUMBER_008", "STATE_FIPS",
    "DECK_COND_058", "SUPERSTRUCTURE_COND_059", "SUBSTRUCTURE_COND_060",
    "CHANNEL_COND_061", "CULVERT_COND_062", "SCOUR_CRITICAL_113",
]}

# NBI code columns where a raw "99" means "miscoded / missing" (per the NBI dictionary).
SENTINEL99 = [
    "HISTORY_037", "DESIGN_LOAD_031", "MEDIAN_CODE_033", "STRUCTURE_FLARED_035",
    "STRUCTURE_KIND_043A", "STRUCTURE_TYPE_043B", "APPR_KIND_044A", "APPR_TYPE_044B",
    "SERVICE_ON_042A", "SERVICE_UND_042B", "BASE_HWY_NETWORK_012", "TOLL_020",
    "FUNCTIONAL_CLASS_026", "STRAHNET_HIGHWAY_100", "TRAFFIC_DIRECTION_102",
    "HIGHWAY_SYSTEM_104", "FEDERAL_LANDS_105", "DECK_STRUCTURE_TYPE_107",
    "NATIONAL_NETWORK_110", "PIER_PROTECTION_111",
]

print(f"{len(LOAD_COLS)} columns to load per state; rare-level threshold = {RARE_THRESHOLD} bridges")

In [ ]:
# ── Load the Notebook 2/3 environmental tables once (kept resident across the state loop) ──
cell_map = pd.read_csv(CELL_MAP, dtype={"STRUCTURE_NUMBER_008": str, "STATE_FIPS": str})
coastal  = pd.read_csv(COASTAL,  dtype={"STRUCTURE_NUMBER_008": str, "STATE_FIPS": str})

# Normalise join keys identically everywhere (raw STRUCTURE_NUMBER_008 is right-justified;
# strip so the merges can never silently miss). Raw padding VARIES across survey years, so
# Notebooks 2/3's dedup on the unstripped key left ~57k whitespace-variant duplicate bridge keys in
# both files; re-dedup on the stripped composite key so every merge is strictly one-to-one
# (otherwise the post-collapse cell/coastal joins multiply rows for those bridges).
for name in ("cell_map", "coastal"):
    d = globals()[name]
    d["STRUCTURE_NUMBER_008"] = d["STRUCTURE_NUMBER_008"].str.strip()
    d["STATE_FIPS"] = d["STATE_FIPS"].str.strip()
    globals()[name] = d.drop_duplicates(KEYS, keep="first")
cell_map, coastal = globals()["cell_map"], globals()["coastal"]

# Climate is 4.9M cell-years; parse the 17 features straight to float32 (a plain float64
# read needs ~0.8 GB and OOMs — the Notebook 2 RAM limit). grid_lat/grid_lon stay float64 so
# they still merge exactly with cell_map. One typed read keeps peak near the ~0.45 GB result.
CLIMATE_FEATURES = [c for c in pd.read_csv(CLIMATE, nrows=0).columns
                    if c not in ("SURVEY_YEAR", "grid_lat", "grid_lon")]
_dtypes = {c: "float32" for c in CLIMATE_FEATURES}
_dtypes.update({"SURVEY_YEAR": "int32", "grid_lat": "float64", "grid_lon": "float64"})
climate = pd.read_csv(CLIMATE, dtype=_dtypes)

# Collapse to CLIMATE NORMALS (per-cell means over 1992-2025). Joining per-year values
# at the kept row's SURVEY_YEAR leaks the observation era: Option A keeps the first-
# event row for failures (median 1996) but the last row for censored bridges (median
# 2025), so yearly climate fingerprints censoring status (measured +0.011 C-index on a
# national Cox arm; +0.09-0.13 on PA tree models). Normals carry only spatial signal.
climate = climate.groupby(["grid_lat", "grid_lon"], as_index=False)[CLIMATE_FEATURES].mean()

print(f"cell_map:  {len(cell_map):,} bridges")
print(f"coastal:   {len(coastal):,} bridges")
print(f"climate:   {len(climate):,} grid cells (normals), {len(CLIMATE_FEATURES)} features")


In [ ]:
# ── Per-state cleaning: derive event / bridge_age / reconstruction + NBI sentinel flags ──
# Mirrors ../ma_bridges/bridge_project.ipynb cell 19 (the MA "rsf_ready" cleaning), verbatim logic.
def clean_state(df):
    # event: 'N' (not applicable) -> 10 so it never trips the "poor" threshold; then <=4.
    cond = ["DECK_COND_058", "SUPERSTRUCTURE_COND_059", "SUBSTRUCTURE_COND_060"]
    for c in cond:
        df[c] = pd.to_numeric(df[c].replace("N", 10), errors="coerce")
    df["event"] = (
        (df["DECK_COND_058"] <= 4) | (df["SUPERSTRUCTURE_COND_059"] <= 4) | (df["SUBSTRUCTURE_COND_060"] <= 4)
    ).astype("int8")

    for c in ["SURVEY_YEAR", "YEAR_BUILT_027", "YEAR_RECONSTRUCTED_106"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # 99 = miscoded/missing on these code columns.
    for c in SENTINEL99:
        if c in df.columns:
            df[c] = df[c].replace(99, np.nan).replace("99", np.nan)

    # 99.99 clearance = "unrestricted" (a real value, not missing) -> keep original, add a flag.
    for c in ["MIN_VERT_CLR_010", "VERT_CLR_OVER_MT_053"]:
        if c in df.columns:
            df[f"{c}_unrestricted"] = (pd.to_numeric(df[c], errors="coerce") == 99.99).astype("int8")

    # skew 99 = "major variation" (real), inventory 999 = "live load insignificant" (real) -> flag, NaN the number.
    if "DEGREES_SKEW_034" in df.columns:
        v = pd.to_numeric(df["DEGREES_SKEW_034"], errors="coerce")
        df["skew_major_variation"] = (v == 99).astype("int8")
        df["DEGREES_SKEW_034"] = v.replace(99, np.nan)
    if "INVENTORY_RATING_066" in df.columns:
        v = pd.to_numeric(df["INVENTORY_RATING_066"], errors="coerce")
        df["inventory_rating_insignificant_load"] = (v == 999).astype("int8")
        df["INVENTORY_RATING_066"] = v.replace(999, np.nan)

    # Reconstruction proxies (national maintenance substitute).
    df["ever_reconstructed"] = (df["YEAR_RECONSTRUCTED_106"] > 0).astype("int8")
    # anchored to REF_YEAR, not the kept row's SURVEY_YEAR: under Option A the kept-row
    # year correlates with event status, so a survey-year covariate carries the era.
    df["years_since_reconstruction"] = np.where(
        df["YEAR_RECONSTRUCTED_106"] > 0, REF_YEAR - df["YEAR_RECONSTRUCTED_106"], np.nan)

    # Time axis.
    df["bridge_age"] = df["SURVEY_YEAR"] - df["YEAR_BUILT_027"]
    df["bridge_age_negative_flag"] = (df["bridge_age"] < 0).astype("int8")
    return df


# ── Option A collapse: one row per bridge = FIRST event row, else LAST observed row ──
# (matches ../ma_bridges/Build_RSF.ipynb cell 1). Idempotent: safe to run per-chunk AND again on the
# concatenated candidates — the second pass finalises the global first-event / last row,
# because each chunk contributes its own first-event and last row per bridge.
def collapse_option_a(df):
    df = df.sort_values(["STRUCTURE_NUMBER_008", "SURVEY_YEAR"])
    event_rows = df[df["event"] == 1].drop_duplicates("STRUCTURE_NUMBER_008", keep="first")
    last_rows  = df.drop_duplicates("STRUCTURE_NUMBER_008", keep="last")
    censored   = last_rows[~last_rows["STRUCTURE_NUMBER_008"].isin(event_rows["STRUCTURE_NUMBER_008"])]
    out = pd.concat([event_rows, censored], ignore_index=True)
    out["time"] = out["bridge_age"].astype(float)
    return out[out["time"] > 0]                   # drop unusable (bad YEAR_BUILT) ages


def process_state(path):
    # Stream the raw file in CHUNK-row blocks and collapse each to per-bridge candidate rows,
    # so a 1.8M-row state (TX) never sits in memory whole. The climate normals and coastal
    # tables are year-independent, so collapsing before the merges is trivially exact.
    # low_memory=False: pandas 3.0's default block-wise parse combined with usecols raises
    # IndexError while composing its mixed-dtype warning (columns whose dtype varies across
    # internal blocks — hit on AR). Memory stays bounded: chunksize already streams the file.
    cand = []
    for chunk in pd.read_csv(path, usecols=LOAD_COLS, dtype=STR_COLS, chunksize=CHUNK,
                             low_memory=False):
        chunk["STRUCTURE_NUMBER_008"] = chunk["STRUCTURE_NUMBER_008"].str.strip()
        chunk["STATE_FIPS"] = chunk["STATE_FIPS"].str.strip()
        cand.append(collapse_option_a(clean_state(chunk)))
    df = collapse_option_a(pd.concat(cand, ignore_index=True))          # global one row per bridge
    df["SURVEY_YEAR"] = df["SURVEY_YEAR"].astype("int32")
    # validate="m:1" asserts each lookup table is unique on its key -> a one-to-one join that
    # can never inflate the one-row-per-bridge panel (raises loudly if a table regains dup keys).
    df = df.merge(cell_map, on=KEYS, how="left", validate="m:1")                        # -> grid_lat, grid_lon
    df = df.merge(climate, on=["grid_lat", "grid_lon"], how="left", validate="m:1")     # climate NORMALS (era-free)
    df = df.merge(coastal, on=KEYS, how="left", validate="m:1")                         # static coastal distance
    return df

print("helpers defined")


In [ ]:
# ── Build the national one-row-per-bridge table ─────────────────────
import time as _t, gc
files = sorted(RAW_DIR.glob("*_bridges.csv"))
print(f"{len(files)} state files")

parts, t0 = [], _t.time()
for i, f in enumerate(files, 1):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        c = process_state(f)
    parts.append(c)
    print(f"[{i:>2}/{len(files)}] {f.stem:<16} {len(c):>7,} bridges  "
          f"{int(c['event'].sum()):>6,} events  ({_t.time()-t0:.0f}s)")
    gc.collect()

bridges = pd.concat(parts, ignore_index=True)
del parts
gc.collect()
bridges.to_csv(OUT_A, index=False)
print(f"\nSaved {OUT_A}: {len(bridges):,} bridges, {bridges.shape[1]} cols")
print(f"Events: {int(bridges['event'].sum()):,} ({bridges['event'].mean()*100:.1f}%)  "
      f"Censored: {int((bridges['event']==0).sum()):,}")
print(f"time range: {bridges['time'].min():.0f}–{bridges['time'].max():.0f} yr")


In [ ]:
# ── National encoding (port of ../ma_bridges/Build_RSF.ipynb cell 2; data-driven rare-level collapse) ──
# Free the environmental tables first (not needed past assembly) and reuse the Option-A table
# in place so the wide one-hot matrix is the only large object. All dummies are int8 (1 byte).
import gc
for _n in ("climate", "cell_map", "coastal"):
    globals().pop(_n, None)
gc.collect()

# Re-runnable: use the in-memory `bridges` from cell 4 if present; otherwise reload it from the
# CSV cell 4 saved (so this cell also works after a kernel restart or a second run of this cell).
if "bridges" not in globals():
    bridges = pd.read_csv(OUT_A, dtype={"STRUCTURE_NUMBER_008": str, "STATE_FIPS": str}, low_memory=False)
df = bridges
del bridges
gc.collect()

def norm_code(series):
    """Collapse float/string spellings of the same NBI code so one-hot can't split them:
    '2.0' / 2.0 / '2' -> '2', 10.0 -> '10', while genuine letter codes ('N','A','2A') pass
    through. A code column reads as float in states where it is all-numeric ('2.0') but as
    object where any row holds a letter ('2'), so a plain .astype(str) would emit two dummy
    columns for the same level (e.g. SURFACE_TYPE_2 and SURFACE_TYPE_2.0)."""
    s = series.astype(str).str.strip()
    num = pd.to_numeric(s, errors="coerce")                  # '2.0' -> 2.0 ; 'N' -> NaN
    is_int = num.notna() & (num == np.floor(num))
    int_str = num.round().astype("Int64").astype("string")   # '2', '10', <NA>
    return s.mask(is_int, int_str)

# Reference (first) level of every one-hot group. get_dummies emits columns in sorted level
# order, so [0] is exactly what drop_first=True would remove. Kept in the RSF matrix (trees
# don't care about the redundant column); dropped in the parametric matrix at the end so
# Cox/AFT/logistic get a full-rank design without the dummy-variable trap.
first_dummies = []

def onehot_rare(df, col, prefix, na_label="Unknown", threshold=RARE_THRESHOLD):
    """One-hot `col`, folding non-null levels rarer than `threshold` into 'Other'.
    Codes are normalised first so '2' and '2.0' collapse to a single dummy."""
    s = norm_code(df[col]).astype(str).str.strip().replace(
        {"nan": na_label, "None": na_label, "N": na_label, "<NA>": na_label})
    vc = s.value_counts()
    rare = vc[vc < threshold].index.difference([na_label])
    s = s.where(~s.isin(rare), "Other")
    dummies = pd.get_dummies(s, prefix=prefix, dtype="int8")
    first_dummies.append(dummies.columns[0])
    return df.drop(columns=[col]).join(dummies)

# DESIGN_LOAD_031 -> ordinal rank (9 -> 7 to stay contiguous; 7/8/A/missing -> NaN as in MA).
df["design_load_ordinal"] = norm_code(df["DESIGN_LOAD_031"]).astype(str).map(
    {"1":1,"2":2,"3":3,"4":4,"5":5,"6":6,"9":7})

# TEMP_STRUCTURE_103 -> binary.
df["TEMP_STRUCTURE_103"] = (df["TEMP_STRUCTURE_103"].astype(str).str.strip() == "T").astype("int8")

# SCOUR_CRITICAL_113: N/U/T/99 are inapplicable-or-unknown -> NaN; rest numeric.
df["SCOUR_CRITICAL_113"] = pd.to_numeric(
    df["SCOUR_CRITICAL_113"].replace({"N": np.nan, "U": np.nan, "T": np.nan, "99": np.nan}), errors="coerce")

# FUNCTIONAL_CLASS_026 -> rural flag + collapsed road class (drop the raw code).
fc = pd.to_numeric(df["FUNCTIONAL_CLASS_026"], errors="coerce")
df["IS_RURAL_026"] = fc.isin([1,2,6,7,8,9]).astype("int8")
road_map = {1:"Interstate",11:"Interstate",2:"Principal_Arterial",12:"Principal_Arterial",
            14:"Principal_Arterial",6:"Minor_Arterial",16:"Minor_Arterial",7:"Collector",
            17:"Collector",8:"Minor_Collector",9:"Local",19:"Local"}
df["ROAD_CLASS_026"] = fc.map(road_map)
df = pd.get_dummies(df, columns=["ROAD_CLASS_026"], prefix="ROAD_CLASS", dtype="int8")
first_dummies.append([c for c in df.columns if c.startswith("ROAD_CLASS_")][0])
df = df.drop(columns=["FUNCTIONAL_CLASS_026"])

# Data-driven one-hots for the structural / operational categoricals (codes normalised).
for col, pfx in [
    ("DECK_STRUCTURE_TYPE_107","DECK_TYPE"), ("SURFACE_TYPE_108A","SURFACE_TYPE"),
    ("MEMBRANE_TYPE_108B","MEMBRANE_TYPE"), ("DECK_PROTECTION_108C","DECK_PROTECTION"),
    ("STRUCTURE_KIND_043A","STRUCTURE_KIND_043A"), ("STRUCTURE_TYPE_043B","STRUCTURE_TYPE_043B"),
    ("SERVICE_ON_042A","SERVICE_ON_042A"), ("SERVICE_UND_042B","SERVICE_UND_042B"),
    ("MEDIAN_CODE_033","MEDIAN_CODE_033"), ("TOLL_020","TOLL_020"),
    ("STRAHNET_HIGHWAY_100","STRAHNET_HIGHWAY_100"), ("TRAFFIC_DIRECTION_102","TRAFFIC_DIRECTION_102"),
    ("PIER_PROTECTION_111","PIER_PROTECTION_111"), ("HISTORY_037","HISTORY"),
    ("MAINTENANCE_021","MAINTENANCE"),
]:
    df = onehot_rare(df, col, pfx)

# STATE one-hot for geography, but KEEP raw STATE_FIPS as an identifier column
# (paired with STRUCTURE_NUMBER_008 it maps every row back to a bridge). NOT normalised —
# FIPS codes are zero-padded strings ('01','06','25') and must stay distinct.
state_dummies = pd.get_dummies(df["STATE_FIPS"], prefix="STATE", dtype="int8")
first_dummies.append(state_dummies.columns[0])          # STATE_01 (AL) = reference level
df = df.join(state_dummies)
del state_dummies

# ── Numeric NBI sentinels / impossible values -> NaN ──────────────────────────
# Fixed sentinel codes ("no restriction / not applicable / miscoded" per the NBI dictionary).
# Listed ONLY where the sentinel is not a physically plausible value for that field — climate
# freeze_thaw=99, ADT=100, STRUCTURE_LEN=99.9 m etc. are REAL values and are left untouched.
SENTINEL_MAP = {
    "HORR_CLR_MT_047":      [99.0, 99.9],            # horizontal clearance "no restriction"
    "APPR_WIDTH_MT_032":    [99.9, 999.9],
    "LEFT_CURB_MT_050A":    [99.9],
    "RIGHT_CURB_MT_050B":   [99.9],
    "ROADWAY_WIDTH_MT_051": [99.9, 999.9],
    "DECK_WIDTH_MT_052":    [99.9, 999.9],
    "VERT_CLR_UND_054B":    [99.0, 99.9, 99.99],
    "LAT_UND_MT_055B":      [99.0, 99.9],
    "LEFT_LAT_UND_MT_056":  [99.0, 99.8, 99.9],
    "OPERATING_RATING_064": [99.0, 99.9],
    "INVENTORY_RATING_066": [99.0, 99.9],            # (999 "insignificant load" flagged in cell 3)
    "DETOUR_KILOS_019":     [999.0],                 # 999 km detour = sentinel
}
for col, bad in SENTINEL_MAP.items():
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").replace(bad, np.nan)

# Range-based impossible values (negatives / absurd magnitudes -> NaN).
def _nan_outside(col, lo=None, hi=None):
    if col not in df.columns:
        return
    v = pd.to_numeric(df[col], errors="coerce")
    if lo is not None: v = v.mask(v < lo)
    if hi is not None: v = v.mask(v > hi)
    df[col] = v

_nan_outside("MAIN_UNIT_SPANS_045",    lo=1, hi=998)     # 0/neg impossible; 999 sentinel
_nan_outside("APPR_SPANS_046",         lo=0, hi=999)     # 0 valid; -1 and 6000 impossible
_nan_outside("TRAFFIC_LANES_UND_028B", lo=0, hi=98)      # 99 all-9s sentinel; 1853 is a typo
_nan_outside("STRUCTURE_LEN_MT_049",   lo=0, hi=40000)   # >40 km impossible (US max ~38 km)
_nan_outside("APPR_WIDTH_MT_032",      lo=0, hi=200)     # >200 m approach width impossible
_nan_outside("ROADWAY_WIDTH_MT_051",   lo=0, hi=200)     # >200 m roadway width impossible
_nan_outside("DECK_WIDTH_MT_052",      lo=0, hi=200)     # >200 m deck width impossible (widest ~80 m)

# Impossible build years -> NaN, and invalidate the age/time derived from them (a handful of
# rows: YEAR_BUILT 0/19 -> time ~2000). Genuinely old bridges (1697, 1764, ...) are kept.
if "YEAR_BUILT_027" in df.columns:
    yb = pd.to_numeric(df["YEAR_BUILT_027"], errors="coerce")
    bad_year = yb < 1690                                 # oldest real US NBI bridge is 1697
    df["YEAR_BUILT_027"] = yb.mask(bad_year)
    for c in ("bridge_age", "time"):
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").mask(bad_year)

# The masking leaves those rows with NaN survival time — unusable for any survival model and
# a violation of the time > 0 contract the training notebook asserts -> drop them entirely.
n_bad_time = int(df["time"].isna().sum())
df = df[df["time"].notna()].reset_index(drop=True)
if n_bad_time:
    print(f"dropped {n_bad_time} rows whose survival time was invalidated (impossible YEAR_BUILT)")

# Reconstruction proxies: rebuild from a cleaned YEAR_RECONSTRUCTED (0 / 9999 / junk -> NaN);
# otherwise the 9999 sentinel propagates into years_since_reconstruction as values like -7974.
if "YEAR_RECONSTRUCTED_106" in df.columns:
    yr = pd.to_numeric(df["YEAR_RECONSTRUCTED_106"], errors="coerce")
    yr = yr.mask((yr < 1600) | (yr > 2025))
    df["YEAR_RECONSTRUCTED_106"] = yr
    df["ever_reconstructed"] = yr.notna().astype("int8")
    # REF_YEAR, not the kept row's SURVEY_YEAR: the kept-row year correlates with event
    # status under Option A, so a survey-year-anchored covariate would carry the era.
    ysr = REF_YEAR - yr
    df["years_since_reconstruction"] = ysr.mask((ysr < 0) | (ysr > 200))

# any residual booleans -> int8 for sksurv.
bcols = df.select_dtypes(include="bool").columns
df[bcols] = df[bcols].astype("int8")

# Drop leakage / administrative / high-NaN / merge-key columns.
# STATE_FIPS + STRUCTURE_NUMBER_008 are kept as identifiers (not model features; they are
# excluded from X at train time).
drop_cols = [
    # leakage: condition ratings and their summaries
    "DECK_COND_058","SUPERSTRUCTURE_COND_059","SUBSTRUCTURE_COND_060",
    "CHANNEL_COND_061","CULVERT_COND_062","OPEN_CLOSED_POSTED_041",
    # transformed originals (YEAR_RECONSTRUCTED_106 == REF_YEAR - years_since_reconstruction
    # after the fixed-year anchor, so keeping it would add an exactly collinear pair)
    "DESIGN_LOAD_031","YEAR_RECONSTRUCTED_106",
    # geography / location identifiers, NOT model features (MA's ../ma_bridges/RSF.ipynb drops LAT_016,
    # LONG_017, KILOPOINT_011 at train time; they stay in us_rsf_data_a.csv for mapping).
    # LAT_016/LONG_017 are also mixed-encoding (decimal AND raw DDMMSS) so unusable as-is.
    "COUNTY_CODE_003","HIGHWAY_DISTRICT_002","PLACE_CODE_004","grid_lat","grid_lon",
    "LAT_016","LONG_017","KILOPOINT_011",
    # administrative / redundant / high-NaN (same as MA ../ma_bridges/Build_RSF.ipynb)
    "RECORD_TYPE_005A","CRITICAL_FACILITY_006B","RAILINGS_036A","TRANSITIONS_036B",
    "APPR_RAIL_036C","APPR_RAIL_END_036D","NAVIGATION_038","VERT_CLR_UND_REF_054A",
    "LAT_UND_REF_055A","BRIDGE_LEN_IND_112","YEAR_ADT_030","OWNER_022","APPR_KIND_044A",
    "APPR_TYPE_044B","DATE_OF_INSPECT_090","VERT_CLR_OVER_MT_053","FEDERAL_LANDS_105",
    "DECK_AREA","MIN_VERT_CLR_010","LRS_INV_ROUTE_013A","SUBROUTE_NO_013B","SURVEY_YEAR",
]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

df.to_csv(OUT_CLEAN, index=False)
print(f"Saved {OUT_CLEAN}: {df.shape[0]:,} rows x {df.shape[1]} cols")

# Parametric variant: identical rows/levels, minus the reference dummy of every one-hot group
# (== drop_first=True, but derived from the SAME encoding pass so the two files can never
# disagree on levels). columns= writes the subset without materialising a second copy in RAM.
param_cols = [c for c in df.columns if c not in set(first_dummies)]
df.to_csv(OUT_PARAM, index=False, columns=param_cols)
print(f"Saved {OUT_PARAM}: {df.shape[0]:,} rows x {len(param_cols)} cols "
      f"(dropped {len(first_dummies)} reference-level dummies)")
print("reference levels dropped:", ", ".join(first_dummies))

print("NOTE: `bridge_age` == `time` by construction (time is derived from age); bridge_age "
      "is excluded from X at train time or it would leak the target.")

In [ ]:
# ── Verification ───────────────────────────────────────────────────────────────
a = pd.read_csv(OUT_A, dtype={"STRUCTURE_NUMBER_008": str, "STATE_FIPS": str}, low_memory=False)
clean = pd.read_csv(OUT_CLEAN, dtype={"STRUCTURE_NUMBER_008": str, "STATE_FIPS": str}, low_memory=False)

print("1. ROW COUNTS")
print(f"   bridges (Option A): {len(a):,}   clean: {len(clean):,}   cols: {clean.shape[1]}")
print(f"   national event rate: {a['event'].mean()*100:.1f}%  ({int(a['event'].sum()):,} events)")

print("\n2. MA REPRODUCTION (FIPS 25) vs standalone MA model (~19,956 bridges, ~3,989 events)")
ma = a[a["STATE_FIPS"] == "25"]
print(f"   MA bridges: {len(ma):,}   MA events: {int(ma['event'].sum()):,}   "
      f"rate: {ma['event'].mean()*100:.1f}%")

print("\n3. MERGE INTEGRITY")
miss_clim = a["freeze_thaw_cycles"].isna().mean() * 100
miss_coast = a["dist_to_coast_km"].isna().mean() * 100
imp = a["location_imputed"]
print(f"   NaN climate: {miss_clim:.2f}%   NaN coastal: {miss_coast:.2f}%   "
      f"location_imputed: {imp.mean()*100:.1f}%")
# Coverage: cell_map/coastal are built valid-coord-FIRST (a bridge is captured from ANY survey
# year with a good coordinate, not just its first row); the ~7.6% with NO coordinate in any
# year are imputed to their COUNTY CENTROID and flagged `location_imputed=1`. So coastal ~0%.
# The residual climate NaN = grid cells not in the Daymet table (water / interior-AK + cells
# the valid-first & imputed maps newly reference) -> closable with an optional Daymet top-up.
c_meas = a.loc[imp == 0, "freeze_thaw_cycles"].isna().mean() * 100
c_imp  = a.loc[imp == 1, "freeze_thaw_cycles"].isna().mean() * 100
print(f"   climate NaN by group: measured {c_meas:.1f}%   imputed {c_imp:.1f}%   (coastal fully covered)")

print("\n4. NO LEAKAGE (must all be False)")
leak = ["DECK_COND_058","SUPERSTRUCTURE_COND_059","SUBSTRUCTURE_COND_060",
        "CHANNEL_COND_061","CULVERT_COND_062","BRIDGE_CONDITION","LOWEST_RATING",
        "OPEN_CLOSED_POSTED_041"]
print("   " + ", ".join(f"{c}={c in clean.columns}" for c in leak))

print("\n5. ENCODING SANITY")
print(f"   STATE_* dummies: {sum(c.startswith('STATE_') for c in clean.columns)}   "
      f"COUNTY_* dummies: {sum(c.startswith('COUNTY_') for c in clean.columns)} (want 0)")
print(f"   ever_reconstructed present: {'ever_reconstructed' in clean.columns}   "
      f"maintenance-history absent: {'had_maintenance' not in clean.columns}   "
      f"location_imputed present: {'location_imputed' in clean.columns}")

print("\n6. PARAMETRIC MATRIX (drop_first variant)")
# Header-only + two-column reads keep this cheap; the full parametric file is ~1 GB.
param_cols_check = pd.read_csv(OUT_PARAM, nrows=0).columns
param_rows = len(pd.read_csv(OUT_PARAM, usecols=["STATE_FIPS"], dtype=str))
dropped = clean.columns.difference(param_cols_check)
extra   = param_cols_check.difference(clean.columns)
only_dummies = all(clean[c].dropna().isin([0, 1]).all() for c in dropped)
print(f"   rows: {param_rows:,} (clean: {len(clean):,} — must match)   "
      f"cols: {len(param_cols_check)} = {clean.shape[1]} - {len(dropped)} reference dummies")
print(f"   no columns beyond clean's: {len(extra) == 0}   "
      f"every dropped column is a 0/1 dummy: {only_dummies}")
print("   dropped:", ", ".join(sorted(dropped)))
assert param_rows == len(clean) and len(extra) == 0 and only_dummies